# Form D API: private placements and exempt offerings

Every Form D notice of an exempt offering, searchable by company, ticker and date, for tracking private raises by public and private companies.

Read it with charts at https://www.quantisnow.com/sec-api/guides/form-d-private-placements-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A company that sells securities under Regulation D files a Form D within 15 days of the first sale. Startups raising venture rounds file them, and so do listed companies doing private placements. In the [filings endpoint](https://www.quantisnow.com/sec-api/filings-search) they are `formType:D`, with amendments as `D/A`.

### Which companies filed a Form D in the last week?

Form D is filed after a private raise. Useful for tracking who is raising money outside a public offering.

```shell
curl -X POST https://www.quantisnow.com/api/v1/filings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:D AND filedAt:[2026-11-17 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 3):

```json
{
  "total": {
    "value": 3,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001934245-26-000004",
      "accessionNo": "0001934245-26-000004",
      "formType": "D",
      "filedAt": "2026-11-20T16:16:50-05:00",
      "cik": "0001934245",
      "companyName": "Newton Golf Company Inc.",
      "ticker": "NWTG",
      "tickers": [
        "NWTG"
      ],
      "items": [],
      "description": "SEC Form D filed by Newton Golf Company Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1934245/000193424526000004/0001934245-26-000004-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-d-filed-by-newton-golf-company-inc-6802290"
    },
    {
      "id": "0001231919-26-001602",
      "accessionNo": "0001231919-26-001602",
      "formType": "D",
      "filedAt": "2026-11-18T16:44:26-05:00",
      "cik": "0001422143",
      "companyName": "Kura Oncology Inc.",
      "ticker": "KURA",
      "tickers": [
        "KURA"
      ],
      "items": [],
      "description": "SEC Form D filed by Kura Oncology Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1422143/000123191926001602/0001231919-26-001602-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-d-filed-by-kura-oncology-inc-6799136"
    }
  ]
}
```

## Listed companies raising privately

Records with a `ticker` are from companies whose stock trades. A Form D from a listed company often means a PIPE, a private sale of shares or convertibles that will dilute holders once resold.

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()

page = search("filings", 'formType:(D OR "D/A") AND filedAt:[2026-11-01 TO 2026-11-23]',
              sort="filedAt:desc")
for f in page["data"]:
    if f["ticker"]:
        print(f["filedAt"][:10], f["ticker"], f["companyName"], f["linkToFilingIndex"])

## In JavaScript

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

async function search(endpoint, query, { size = 50, sort, from = 0 } = {}) {
  const res = await fetch(BASE + "/" + endpoint, {
    method: "POST",
    headers: {
      Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({ query, size, from, sort }),
  });
  if (!res.ok) throw new Error((await res.json()).error.message);
  return res.json();
}

// Biotech raises this month, by name
const page = await search("filings", "formType:D AND filedAt:[2026-11-01 TO 2026-11-23]", { sort: "filedAt:desc" });
const biotech = page.data.filter((f) => /therapeutics|pharma|bio/i.test(f.companyName));
console.log(biotech.map((f) => f.companyName + " " + f.filedAt.slice(0, 10)));
```

The amount raised, the investors and the exemption claimed are inside the Form D itself; the filing index link has the document. Structured offering details are on the list of datasets we're adding.